# Cue-Locked EEG Activation EDA

For subjects A1–A5, this notebook separately aggregates every left-hand (`769`) and right-hand (`770`) cue from online Runs 3–6. Each head-shaped topographic heatmap shows the average baseline-normalized 8–30 Hz power during the motor-imagery stimulus interval for all 27 scalp EEG electrodes.

- **0 seconds:** the motor-imagery cue appears.
- **Negative values:** event-related desynchronization (less 8–30 Hz power than baseline).
- **Positive values:** event-related synchronization (more 8–30 Hz power than baseline).

The final intensity at each electrode is averaged across trials of the indicated cue and across the 0-to-5-second stimulus interval. These maps visualize neural activity associated with the stimuli; they are not classifier predictions or per-trial accuracy.

In [ ]:
from pathlib import Path
import re
import warnings

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd

mne.set_log_level("ERROR")
warnings.filterwarnings("ignore", category=RuntimeWarning)

search_roots = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
PROJECT_ROOT = next(
    (
        root
        for root in search_roots
        if (root / "data" / "processed" / "Signals").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Run this notebook from inside the bci_cleaning project")

SIGNALS_ROOT = PROJECT_ROOT / "data" / "processed" / "Signals"
NON_SCALP_CHANNELS = {"EOG1", "EOG2", "EOG3", "EMGg", "EMGd"}
CUE_EVENT_ID = {"Left-hand cue": 769, "Right-hand cue": 770}
FREQUENCY_BAND_HZ = (8.0, 30.0)
TMIN, TMAX = -2.0, 5.0


def subject_sort_key(path):
    match = re.fullmatch(r"([A-Z])(\d+)", path.name)
    return (match.group(1), int(match.group(2))) if match else (path.name, 0)


subject_directories = sorted(
    (
        subject_dir
        for dataset_dir in SIGNALS_ROOT.iterdir()
        if dataset_dir.is_dir()
        for subject_dir in dataset_dir.iterdir()
        if subject_dir.is_dir()
        and re.fullmatch(r"[ABC]\d+", subject_dir.name)
        and list(subject_dir.glob("*_R3_onlineT.gdf"))
    ),
    key=subject_sort_key,
)
selected_subjects = subject_directories[:5]

print("Selected subjects:", [path.name for path in selected_subjects])
for subject_dir in selected_subjects:
    run_files = sorted(subject_dir.glob("*_R[3-6]_onlineT.gdf"))
    print(f"{subject_dir.name}: {len(run_files)} online files")

In [ ]:
def aggregate_subject_activation(subject_dir):
    """Return mean cue-locked 8–30 Hz power change for one participant."""
    run_files = sorted(subject_dir.glob("*_R[3-6]_onlineT.gdf"))
    if not run_files:
        raise FileNotFoundError(f"No online GDF runs found for {subject_dir.name}")

    normalized_trials = {
        "Left-hand cue": [],
        "Right-hand cue": [],
    }
    channel_names = None
    epoch_times = None
    cue_counts = {"Left-hand cue": 0, "Right-hand cue": 0}

    for run_file in run_files:
        raw = mne.io.read_raw_gdf(
            run_file,
            preload=True,
            verbose="ERROR",
        )
        scalp_channels = [
            channel
            for channel in raw.ch_names
            if channel not in NON_SCALP_CHANNELS
        ]
        raw.pick(scalp_channels)

        if channel_names is None:
            channel_names = raw.ch_names.copy()
        elif raw.ch_names != channel_names:
            raise ValueError(f"Channel order differs in {run_file.name}")

        events, _ = mne.events_from_annotations(
            raw,
            event_id={"769": 769, "770": 770},
            verbose="ERROR",
        )
        if not len(events):
            raw.close()
            continue

        cue_counts["Left-hand cue"] += int((events[:, 2] == 769).sum())
        cue_counts["Right-hand cue"] += int((events[:, 2] == 770).sum())

        # Filter continuously to avoid edge artifacts at individual epoch boundaries.
        raw.filter(
            FREQUENCY_BAND_HZ[0],
            FREQUENCY_BAND_HZ[1],
            method="fir",
            phase="zero",
            verbose="ERROR",
        )
        raw.apply_hilbert(envelope=True, verbose="ERROR")

        epochs = mne.Epochs(
            raw,
            events,
            event_id=CUE_EVENT_ID,
            tmin=TMIN,
            tmax=TMAX,
            baseline=None,
            preload=True,
            decim=8,
            reject_by_annotation=True,
            verbose="ERROR",
        )
        power = np.square(epochs.get_data(copy=False))
        baseline_mask = epochs.times < 0
        baseline_power = power[:, :, baseline_mask].mean(axis=2, keepdims=True)
        baseline_power = np.maximum(baseline_power, np.finfo(float).eps)
        normalized_power = 100 * (power / baseline_power - 1)
        for cue_name, cue_code in CUE_EVENT_ID.items():
            cue_mask = epochs.events[:, 2] == cue_code
            if cue_mask.any():
                normalized_trials[cue_name].append(normalized_power[cue_mask])

        epoch_times = epochs.times.copy()
        raw.close()

    if not all(normalized_trials.values()):
        raise ValueError(f"Both cue types were not found for {subject_dir.name}")

    trials_by_cue = {
        cue_name: np.concatenate(cue_trials, axis=0)
        for cue_name, cue_trials in normalized_trials.items()
    }
    all_trials = np.concatenate(list(trials_by_cue.values()), axis=0)
    return {
        "subject": subject_dir.name,
        "activation": np.nanmean(all_trials, axis=0),
        "activation_by_cue": {
            cue_name: np.nanmean(cue_trials, axis=0)
            for cue_name, cue_trials in trials_by_cue.items()
        },
        "trial_count_by_cue": {
            cue_name: len(cue_trials)
            for cue_name, cue_trials in trials_by_cue.items()
        },
        "times": epoch_times,
        "channels": channel_names,
        "n_trials": len(all_trials),
        "left_cues": cue_counts["Left-hand cue"],
        "right_cues": cue_counts["Right-hand cue"],
        "n_runs": len(run_files),
    }

In [ ]:
subject_activation = []
for subject_dir in selected_subjects:
    print(f"Processing {subject_dir.name}...")
    subject_activation.append(aggregate_subject_activation(subject_dir))

activation_summary = pd.DataFrame(
    {
        "SUJ_ID": result["subject"],
        "online_runs": result["n_runs"],
        "left_cues": result["left_cues"],
        "right_cues": result["right_cues"],
        "aggregated_trials": result["n_trials"],
        "electrodes": len(result["channels"]),
    }
    for result in subject_activation
)
display(activation_summary)

In [ ]:
# Average each cue type and electrode over only the 0-to-5-second stimulus interval.
for result in subject_activation:
    stimulus_mask = (result["times"] >= 0) & (result["times"] <= 5)
    result["stimulus_intensity_by_cue"] = {
        cue_name: np.nanmean(activation[:, stimulus_mask], axis=1)
        for cue_name, activation in result["activation_by_cue"].items()
    }

# Use one robust, symmetric scale so colors are comparable across subjects.
all_intensities = np.concatenate([
    intensity
    for result in subject_activation
    for intensity in result["stimulus_intensity_by_cue"].values()
])
color_limit = np.nanpercentile(np.abs(all_intensities), 98)

montage = mne.channels.make_standard_montage("standard_1020")
topomap_info = mne.create_info(
    subject_activation[0]["channels"],
    sfreq=1.0,
    ch_types="eeg",
)
topomap_info.set_montage(montage, on_missing="raise")

cue_order = ["Left-hand cue", "Right-hand cue"]
fig, axes = plt.subplots(
    len(subject_activation),
    len(cue_order),
    figsize=(12, 24),
    constrained_layout=True,
)

for row, result in enumerate(subject_activation):
    for column, cue_name in enumerate(cue_order):
        ax = axes[row, column]
        image, _ = mne.viz.plot_topomap(
            result["stimulus_intensity_by_cue"][cue_name],
            topomap_info,
            axes=ax,
            show=False,
            sensors="ko",
            names=result["channels"],
            contours=7,
            outlines="head",
            extrapolate="head",
            cmap="RdBu_r",
            vlim=(-color_limit, color_limit),
            res=256,
        )
        ax.set_title(
            f"{result['subject']} — {cue_name}\n"
            f"{result['trial_count_by_cue'][cue_name]} stimulus events",
            fontsize=12,
        )

colorbar = fig.colorbar(image, ax=axes, shrink=0.6, pad=0.02)
colorbar.set_label(
    "Average 8–30 Hz power change during stimulus (%)",
    fontsize=11,
)
fig.suptitle(
    "Left- vs. Right-Hand Cue-Evoked EEG Activation",
    fontsize=17,
)
plt.show()

In [ ]:
# Create equal-weight grand-average scalp maps for all 87 participants.
if len(subject_directories) != 87:
    raise ValueError(
        f"Expected 87 participant directories, found {len(subject_directories)}"
    )

# Reuse the five results already calculated above, then process the other 82.
results_by_subject = {
    result["subject"]: result for result in subject_activation
}
for position, subject_dir in enumerate(subject_directories, start=1):
    if subject_dir.name not in results_by_subject:
        print(f"[{position:02d}/87] Processing {subject_dir.name}...")
        results_by_subject[subject_dir.name] = aggregate_subject_activation(
            subject_dir
        )

all_subject_activation = [
    results_by_subject[subject_dir.name]
    for subject_dir in subject_directories
]

# Reduce each participant to one electrode value per cue before averaging
# participants, so participants with fewer recorded trials receive equal weight.
for result in all_subject_activation:
    stimulus_mask = (result["times"] >= 0) & (result["times"] <= 5)
    result["stimulus_intensity_by_cue"] = {
        cue_name: np.nanmean(activation[:, stimulus_mask], axis=1)
        for cue_name, activation in result["activation_by_cue"].items()
    }

cue_order = ["Left-hand cue", "Right-hand cue"]
grand_average_by_cue = {
    cue_name: np.nanmean(
        np.stack([
            result["stimulus_intensity_by_cue"][cue_name]
            for result in all_subject_activation
        ]),
        axis=0,
    )
    for cue_name in cue_order
}

grand_values = np.concatenate(list(grand_average_by_cue.values()))
grand_color_limit = np.nanpercentile(np.abs(grand_values), 98)

grand_montage = mne.channels.make_standard_montage("standard_1020")
grand_topomap_info = mne.create_info(
    all_subject_activation[0]["channels"],
    sfreq=1.0,
    ch_types="eeg",
)
grand_topomap_info.set_montage(grand_montage, on_missing="raise")

fig, axes = plt.subplots(1, 2, figsize=(13, 6), constrained_layout=True)
for ax, cue_name in zip(axes, cue_order):
    image, _ = mne.viz.plot_topomap(
        grand_average_by_cue[cue_name],
        grand_topomap_info,
        axes=ax,
        show=False,
        sensors="ko",
        names=all_subject_activation[0]["channels"],
        contours=7,
        outlines="head",
        extrapolate="head",
        cmap="RdBu_r",
        vlim=(-grand_color_limit, grand_color_limit),
        res=256,
    )
    total_trials = sum(
        result["trial_count_by_cue"][cue_name]
        for result in all_subject_activation
    )
    ax.set_title(
        f"{cue_name}\n87 participants, {total_trials:,} stimulus events",
        fontsize=13,
    )

colorbar = fig.colorbar(image, ax=axes, shrink=0.8, pad=0.03)
colorbar.set_label(
    "Grand-average 8–30 Hz power change during stimulus (%)",
    fontsize=11,
)
fig.suptitle(
    "Grand-Average Cue-Evoked EEG Activation Across All Participants",
    fontsize=16,
)
plt.show()

grand_average_summary = pd.DataFrame(
    {
        "cue": cue_order,
        "participants": [len(all_subject_activation)] * 2,
        "stimulus_events": [
            sum(
                result["trial_count_by_cue"][cue_name]
                for result in all_subject_activation
            )
            for cue_name in cue_order
        ],
    }
)
display(grand_average_summary)

In [ ]:
# Self-contained analysis: grand-average cue maps split by participant gender.
# Dataset coding: SUJ_gender 1 = Man, 2 = Woman.
from pathlib import Path
import re
import warnings

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd

mne.set_log_level("ERROR")
warnings.filterwarnings("ignore", category=RuntimeWarning)

gender_search_roots = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
gender_project_root = next(
    (
        root
        for root in gender_search_roots
        if (root / "data" / "processed" / "Signals").is_dir()
    ),
    None,
)
if gender_project_root is None:
    raise RuntimeError("Run this cell from inside the bci_cleaning project")

gender_signals_root = gender_project_root / "data" / "processed" / "Signals"
gender_profile_path = (
    gender_project_root / "data" / "processed" / "Perfomances_cleaned.csv"
)
gender_profile = pd.read_csv(gender_profile_path, sep=";")
gender_lookup = gender_profile.set_index("SUJ_ID")["SUJ_gender"].astype(int)

gender_non_scalp = {"EOG1", "EOG2", "EOG3", "EMGg", "EMGd"}
gender_cue_codes = {"Left-hand cue": 769, "Right-hand cue": 770}


def gender_subject_sort_key(path):
    match = re.fullmatch(r"([A-Z])(\d+)", path.name)
    return (match.group(1), int(match.group(2))) if match else (path.name, 0)


gender_subject_directories = sorted(
    (
        subject_dir
        for dataset_dir in gender_signals_root.iterdir()
        if dataset_dir.is_dir()
        for subject_dir in dataset_dir.iterdir()
        if subject_dir.is_dir()
        and re.fullmatch(r"[ABC]\d+", subject_dir.name)
        and list(subject_dir.glob("*_R3_onlineT.gdf"))
    ),
    key=gender_subject_sort_key,
)
if len(gender_subject_directories) != 87:
    raise ValueError(
        f"Expected 87 participant directories, found "
        f"{len(gender_subject_directories)}"
    )


def extract_gender_cue_activation(subject_dir):
    """Estimate each electrode's raw-EEG activation likelihood per cue."""
    cue_trial_activation = {cue_name: [] for cue_name in gender_cue_codes}
    channel_names = None

    for run_file in sorted(subject_dir.glob("*_R[3-6]_onlineT.gdf")):
        raw = mne.io.read_raw_gdf(
            run_file,
            preload=True,
            verbose="ERROR",
        )
        scalp_channels = [
            channel for channel in raw.ch_names
            if channel not in gender_non_scalp
        ]
        raw.pick(scalp_channels)
        if channel_names is None:
            channel_names = raw.ch_names.copy()
        elif raw.ch_names != channel_names:
            raise ValueError(f"Channel order differs in {run_file.name}")

        events, _ = mne.events_from_annotations(
            raw,
            event_id={"769": 769, "770": 770},
            verbose="ERROR",
        )
        if not len(events):
            raw.close()
            continue

        epochs = mne.Epochs(
            raw,
            events,
            event_id=gender_cue_codes,
            tmin=-2.0,
            tmax=5.0,
            baseline=(-2.0, 0.0),
            preload=True,
            reject_by_annotation=True,
            verbose="ERROR",
        )

        baseline_mask = epochs.times < 0
        stimulus_mask = (epochs.times >= 0) & (epochs.times <= 5)
        raw_voltage_uv = epochs.get_data(copy=False) * 1e6
        baseline_rms = np.sqrt(np.nanmean(
            np.square(raw_voltage_uv[:, :, baseline_mask]), axis=2
        ))
        stimulus_rms = np.sqrt(np.nanmean(
            np.square(raw_voltage_uv[:, :, stimulus_mask]), axis=2
        ))
        trial_is_active = (
            np.isfinite(stimulus_rms)
            & np.isfinite(baseline_rms)
            & (stimulus_rms > baseline_rms)
        )

        for cue_name, cue_code in gender_cue_codes.items():
            cue_mask = epochs.events[:, 2] == cue_code
            if cue_mask.any():
                cue_trial_activation[cue_name].append(
                    trial_is_active[cue_mask]
                )
        raw.close()

    if not all(cue_trial_activation.values()):
        raise ValueError(f"Both cue types were not found for {subject_dir.name}")

    trials_by_cue = {
        cue_name: np.concatenate(trials, axis=0)
        for cue_name, trials in cue_trial_activation.items()
    }
    return {
        "SUJ_ID": subject_dir.name,
        "SUJ_gender": int(gender_lookup.loc[subject_dir.name]),
        "channels": channel_names,
        "activation_probability_by_cue": {
            cue_name: 100 * np.nanmean(trials, axis=0)
            for cue_name, trials in trials_by_cue.items()
        },
        "trial_count_by_cue": {
            cue_name: len(trials)
            for cue_name, trials in trials_by_cue.items()
        },
    }


gender_subject_results = []
for position, subject_dir in enumerate(gender_subject_directories, start=1):
    print(f"[{position:02d}/87] Processing {subject_dir.name}...")
    gender_subject_results.append(extract_gender_cue_activation(subject_dir))

gender_codes = [1, 2]
gender_labels = {1: "Gender 1 (Man)", 2: "Gender 2 (Woman)"}
gender_cue_order = ["Left-hand cue", "Right-hand cue"]
gender_grand_average = {}
for gender_code in gender_codes:
    gender_members = [
        result
        for result in gender_subject_results
        if result["SUJ_gender"] == gender_code
    ]
    for cue_name in gender_cue_order:
        gender_grand_average[(gender_code, cue_name)] = np.nanmean(
            np.stack([
                result["activation_probability_by_cue"][cue_name]
                for result in gender_members
            ]),
            axis=0,
        )

gender_all_probabilities = np.concatenate(
    list(gender_grand_average.values())
)
gender_probability_min = np.floor(np.nanmin(gender_all_probabilities))
gender_probability_max = np.ceil(np.nanmax(gender_all_probabilities))
gender_probability_contours = np.linspace(
    gender_probability_min, gender_probability_max, 8
)

gender_montage = mne.channels.make_standard_montage("standard_1020")
gender_topomap_info = mne.create_info(
    gender_subject_results[0]["channels"],
    sfreq=1.0,
    ch_types="eeg",
)
gender_topomap_info.set_montage(gender_montage, on_missing="raise")

fig, axes = plt.subplots(2, 2, figsize=(15, 13), constrained_layout=True)
gender_summary_rows = []
for row, gender_code in enumerate(gender_codes):
    members = [
        result
        for result in gender_subject_results
        if result["SUJ_gender"] == gender_code
    ]
    for column, cue_name in enumerate(gender_cue_order):
        ax = axes[row, column]
        activation_probability = gender_grand_average[
            (gender_code, cue_name)
        ]
        electrode_labels = [
            f"{channel}\n{probability:.0f}%"
            for channel, probability in zip(
                gender_subject_results[0]["channels"],
                activation_probability,
            )
        ]
        image, _ = mne.viz.plot_topomap(
            activation_probability,
            gender_topomap_info,
            axes=ax,
            show=False,
            sensors="ko",
            names=electrode_labels,
            contours=gender_probability_contours,
            outlines="head",
            extrapolate="head",
            cmap="YlOrRd",
            vlim=(gender_probability_min, gender_probability_max),
            res=256,
        )
        total_trials = sum(
            result["trial_count_by_cue"][cue_name]
            for result in members
        )
        ax.set_title(
            f"{gender_labels[gender_code]} — {cue_name}\n"
            f"{len(members)} participants, {total_trials:,} events",
            fontsize=12,
        )
        gender_summary_rows.append(
            {
                "gender": gender_labels[gender_code],
                "cue": cue_name,
                "participants": len(members),
                "stimulus_events": total_trials,
            }
        )

colorbar = fig.colorbar(image, ax=axes, shrink=0.7, pad=0.03)
colorbar.set_label(
    "Trials with stimulus RMS greater than pre-cue RMS (%)",
    fontsize=11,
)
fig.suptitle(
    "Raw-EEG Electrode Activation Likelihood by Gender and Cue",
    fontsize=16,
)
plt.show()

gender_grand_average_summary = pd.DataFrame(gender_summary_rows)
display(gender_grand_average_summary)